In [1]:
# Q1 NLP Preprocessing Pipeline
import nltk
from nltk.tokenize import wordpunct_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer


def tokenize_text(text):
    """Split the text into word and punctuation tokens."""
    return wordpunct_tokenize(text)


def drop_stopwords(token_list):
    """Keep only tokens that are not English stopwords."""
    english_stops = set(stopwords.words("english"))
    kept = []
    for tok in token_list:
        if tok.lower() not in english_stops:
            kept.append(tok)
    return kept


def stem_tokens(token_list):
    """Reduce each token to its Porter stem."""
    porter = PorterStemmer()
    return list(map(porter.stem, token_list))


def run_pipeline(text):
    raw_tokens = tokenize_text(text)
    clean_tokens = drop_stopwords(raw_tokens)
    root_forms = stem_tokens(clean_tokens)

    results = {
        "Original Tokens": raw_tokens,
        "Tokens Without Stopwords": clean_tokens,
        "Stemmed Words": root_forms,
    }
    for title, values in results.items():
        print(f"{title}:\n{values}\n")


if __name__ == "__main__":
    nltk.download("stopwords", quiet=True)
    input_text = "NLP techniques are used in virtual assistants like Alexa and Siri."
    run_pipeline(input_text)


Original Tokens:
['NLP', 'techniques', 'are', 'used', 'in', 'virtual', 'assistants', 'like', 'Alexa', 'and', 'Siri', '.']

Tokens Without Stopwords:
['NLP', 'techniques', 'used', 'virtual', 'assistants', 'like', 'Alexa', 'Siri', '.']

Stemmed Words:
['nlp', 'techniqu', 'use', 'virtual', 'assist', 'like', 'alexa', 'siri', '.']



In [2]:
# Q2 Named Entity Recognition with SpaCy
import spacy


def find_entities(text, model_name="en_core_web_sm"):
    """Run spaCy NER and return (text, label, start, end) tuples."""
    language_model = spacy.load(model_name)
    parsed = language_model(text)
    return [(e.text, e.label_, e.start_char, e.end_char) for e in parsed.ents]


def show_entities(entity_rows):
    print("Named Entities:")
    for idx, (span, tag, begin, finish) in enumerate(entity_rows, start=1):
        print(f"{idx}. Entity: {span} | Label: {tag} | Start: {begin} | End: {finish}")


if __name__ == "__main__":
    input_text = ("LeBron James as the NBA player of the United States in Laker "
                  "and won the NBA Final MVP in 2020.")
    show_entities(find_entities(input_text))


Named Entities:
1. Entity: LeBron James | Label: PERSON | Start: 0 | End: 12
2. Entity: NBA | Label: ORG | Start: 20 | End: 23
3. Entity: the United States | Label: GPE | Start: 34 | End: 51
4. Entity: Laker | Label: GPE | Start: 55 | End: 60
5. Entity: MVP | Label: ORG | Start: 83 | End: 86
6. Entity: 2020 | Label: DATE | Start: 90 | End: 94


In [3]:
# Q3 Scaled Dot-Product Attention
import numpy as np


def stable_softmax(matrix):
    """Row-wise softmax; subtracting the row max avoids overflow."""
    shifted = matrix - matrix.max(axis=-1, keepdims=True)
    exps = np.exp(shifted)
    return exps / exps.sum(axis=-1, keepdims=True)


def scaled_dot_product_attention(queries, keys, values):
    """Attention(Q, K, V) = softmax(Q K^T / sqrt(d_k)) V"""
    key_dim = keys.shape[-1]
    similarity = queries @ keys.T            # step 1: dot products
    similarity = similarity / np.sqrt(key_dim)  # step 2: scale
    weights = stable_softmax(similarity)     # step 3: softmax
    context = weights @ values               # step 4: weighted sum of V
    return weights, context


if __name__ == "__main__":
    query_mat = np.array([[1, 0, 1, 0],
                          [0, 1, 0, 1]], dtype=float)
    key_mat = query_mat.copy()               # K is the same as Q here
    value_mat = np.arange(1, 9, dtype=float).reshape(2, 4)  # [[1,2,3,4],[5,6,7,8]]

    attn, result = scaled_dot_product_attention(query_mat, key_mat, value_mat)

    np.set_printoptions(precision=4, suppress=True)
    print("Attention Weights:")
    print(attn)
    print("\nFinal Output:")
    print(result)


Attention Weights:
[[0.7311 0.2689]
 [0.2689 0.7311]]

Final Output:
[[2.0758 3.0758 4.0758 5.0758]
 [3.9242 4.9242 5.9242 6.9242]]
